# NetworkX Integration

This notebook demonstrates how to use the NetworkX adapter to solve
MaxCut with QAOA and estimate quantum resource requirements.

**Requirements:** `pip install qdk-pythonic[networkx]`

## 1. MaxCut on a Random Graph

In [ ]:
import networkx as nx

from qdk_pythonic.adapters.networkx_adapter import (
    compare_qaoa_depths,
    solve_maxcut,
)

G = nx.random_regular_graph(d=3, n=10, seed=42)
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

result = solve_maxcut(G, p=2)
print(f"\nQAOA MaxCut (p=2):")
print(f"  Qubits:           {result['n_qubits']}")
print(f"  Total gates:      {result['total_gates']}")
print(f"  Circuit depth:    {result['depth']}")
print(f"  Max possible cut: {result['max_possible_cut']}")

## 2. QAOA Depth Comparison

More QAOA layers give better approximation ratios but cost more gates.

In [ ]:
print(f"{'p':>4} {'Gates':>10} {'Depth':>8} {'Qubits':>8}")
print("-" * 34)

comparisons = compare_qaoa_depths(G, p_values=[1, 2, 3, 4, 5])
for r in comparisons:
    print(f"{r['p']:>4} {r['total_gates']:>10} {r['depth']:>8} {r['n_qubits']:>8}")

## 3. Scaling with Graph Size

In [ ]:
print(
    f"{'Nodes':>6} {'Edges':>6} {'Gates (p=1)':>12} {'Gates (p=3)':>12} "
    f"{'Depth (p=1)':>12} {'Depth (p=3)':>12}"
)
print("-" * 66)

for n in [6, 10, 14, 20, 30]:
    G_s = nx.random_regular_graph(d=3, n=n, seed=42)
    r1 = solve_maxcut(G_s, p=1)
    r3 = solve_maxcut(G_s, p=3)
    print(
        f"{n:>6} {G_s.number_of_edges():>6} "
        f"{r1['total_gates']:>12} {r3['total_gates']:>12} "
        f"{r1['depth']:>12} {r3['depth']:>12}"
    )

## 4. Weighted Graph

In [ ]:
G_w = nx.Graph()
G_w.add_weighted_edges_from([
    (0, 1, 2.0), (1, 2, 1.5), (2, 3, 3.0), (3, 0, 1.0), (0, 2, 0.5),
])

r_w = solve_maxcut(G_w, p=2)
print(f"Weighted graph: {G_w.number_of_nodes()} nodes, {G_w.number_of_edges()} edges")
print(f"  Max possible cut:  {r_w['max_possible_cut']}")
print(f"  Qubits:            {r_w['n_qubits']}")
print(f"  Total gates:       {r_w['total_gates']}")

## 5. Benchmark Graph Families

Test QAOA performance on well-known graph families.

In [ ]:
graphs = {
    "Petersen": nx.petersen_graph(),
    "Cycle(12)": nx.cycle_graph(12),
    "Complete(6)": nx.complete_graph(6),
    "Grid(3x4)": nx.grid_2d_graph(3, 4),
    "Random regular(3,12)": nx.random_regular_graph(3, 12, seed=42),
}

print(
    f"{'Graph':<25} {'Nodes':>6} {'Edges':>6} "
    f"{'Qubits':>7} {'Gates(p=1)':>11} {'Gates(p=3)':>11}"
)
print("-" * 72)

for name, G_b in graphs.items():
    r1 = solve_maxcut(G_b, p=1)
    r3 = solve_maxcut(G_b, p=3)
    print(
        f"{name:<25} {G_b.number_of_nodes():>6} {G_b.number_of_edges():>6} "
        f"{r1['n_qubits']:>7} {r1['total_gates']:>11} {r3['total_gates']:>11}"
    )

## 6. Inspect Generated Q#

In [ ]:
G_small = nx.cycle_graph(4)
r_small = solve_maxcut(G_small, p=1)

print("--- Q# ---")
print(r_small['circuit'].to_qsharp())
print("\n--- OpenQASM 3.0 ---")
print(r_small['circuit'].to_openqasm())